In [0]:
from pyspark.sql import functions as F

inventory = spark.table("workspace.silver.inventory")
products = spark.table("workspace.silver.products")

inventory_kpi = (
    inventory
    .join(
        products.select("product_id", "product_name", "category"),
        "product_id",
        "left"
    )
    .withColumn(
        "available_after_reservation",
        F.col("quantity_available") - F.col("quantity_reserved")
    )
    .withColumn(
        "inventory_status",
        F.when(F.col("available_after_reservation") <= 10, "LOW")
         .otherwise("HEALTHY")
    )
)

(
    inventory_kpi.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.inventory_kpi")
)

display(inventory_kpi)